## Process
- Load CNAF file
- Clean data and first mapping for bdd injection
- Drop duplicates
- Add default column values
- Output to CSV

## Encoding
CNAF -> ascii

In [ ]:
import os
import csv
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# clean_cnaf_lib imports utils.data_utils, which lives at the data/ root: make that root
# importable first, since this notebook runs from its own directory.
try:
    import utils.data_utils  # noqa: F401
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break

# All the DataFrame processing lives next to this notebook, in clean_cnaf_lib.py, so it
# can be unit tested - see test_clean_cnaf_lib.py.
import clean_cnaf_lib as cnaf

load_dotenv()

cnaf_input_filepath = os.environ['CNAF_PATHFILE_2026']
base_output_filepath = os.environ['DB_CNAF_EXPORT_2026']
qf_batch_input_filepath = os.environ['QF_BATCH_INPUT_PATHFILE_2026']
qf_batch_output_filepath = os.environ['QF_BATCH_OUTPUT_PATHFILE_2026']

# INSEE COG countries & territories, used to turn CNAF's PAYSNAIDOS label into a COG code.
# Defaults to the copy shipped next to this notebook when the env var isn't set.
cog_pays_input_filepath = os.environ.get(
    'COG_PAYS_PATHFILE_2026', str(Path.cwd() / 'v_pays_territoire_2026.csv'))


In [ ]:
# CNAF
cnaf_column_type = {
    'CODORG': 'str',
    'MATRICULE': 'str',
    'QUALDOS': 'str',
    'RESPDOS': 'str',
    'NOMNAIDOS': 'str',
    'PRENOMDOS': 'str',
    'DTNAIDOS': 'str',
    'SEXDOS': 'str',
    'COMMUNENAIDOS': 'str',
    'PAYSNAIDOS': 'str',
    'ORIGINESELECTION': 'str',
    'NOMCOMPLET': 'str',
    'ADRLIG1DESTDOS': 'str',
    'ADRLIG2DESTDOS': 'str',
    'ADRLIG3DESTDOS': 'str',
    'ADRLIG4DESTDOS': 'str',
    'ADRLIG5DESTDOS': 'str',
    'ADRLIG6DESTDOS': 'str',
    'NUMINSEE': 'str',
    'ADRMAIL': 'str',
    'NUMTEL': 'str',
    'NOMENF': 'str',
    'PRENOMENF': 'str',
    'DTNAIENF': 'str',
    'SEXENF': 'str',
}

cnaf_df = pd.read_csv(cnaf_input_filepath, encoding='ascii', on_bad_lines='skip', sep=';', quoting=csv.QUOTE_NONE, dtype=cnaf_column_type, engine="c", keep_default_na=False, header=1)


In [ ]:
# delete last row (it is not a valid row) & clean white spaces within all columns
cnaf_df = cnaf.clean_raw_cnaf(cnaf_df)

In [ ]:
# Explode postal code & commune from initial column containing both
cnaf_df = cnaf.split_postal_code_and_commune(cnaf_df)

In [ ]:
# Clean extra white spaces
cnaf_df = cnaf.normalize_full_name_spacing(cnaf_df)

In [ ]:
# map CNAF columns to the PSP schema (see cnaf.CNAF_COLUMN_MAPPING)
df_psp_mapped_cnaf = cnaf.map_cnaf_columns(cnaf_df)

In [ ]:
# Allocataire missing phone number
df_psp_mapped_cnaf = cnaf.clear_placeholder_phone_numbers(df_psp_mapped_cnaf)

# Allocataire's qualite
df_psp_mapped_cnaf = cnaf.normalize_allocataire_qualite(df_psp_mapped_cnaf)

# Additionnal address details & allocataire's street address
df_psp_mapped_cnaf = cnaf.build_allocataire_address_fields(df_psp_mapped_cnaf)

# Organism & situation - CNAF now flags the category itself, no more DOB+name guessing
df_psp_mapped_cnaf = cnaf.set_organisme_and_situation(df_psp_mapped_cnaf)

In [ ]:
# Format date_naissance to datetime python object for processing
df_psp_mapped_cnaf = cnaf.parse_beneficiary_birthdate(df_psp_mapped_cnaf)

In [ ]:
# Build the qf-batch input: only ARS-origin rows carry the allocataire pivot identity
# needed for quotient_familial. One call per household, not per child, since several
# beneficiary rows can share the same allocataire. The 6-17 ans window (cnaf.QF_DOB_MIN /
# cnaf.QF_DOB_MAX) is applied here so we never spend a quotient_familial call on a
# household with no child in the window.

# TODO matricule + code org en pivot suffisant ? A check en db
df_qf_allocataires, df_qf_route = cnaf.select_qf_route_allocataires(df_psp_mapped_cnaf)
df_qf_allocataires = cnaf.format_qf_identity_fields(df_qf_allocataires)

# code_pays_naissance: PAYSNAIDOS holds the country *label* (FRANCE, MAROC, PORTUGAL...),
# not a code - mapped here to its INSEE COG code through v_pays_territoire (France -> 99100,
# Maroc -> 99350...).
df_cog_pays = pd.read_csv(cog_pays_input_filepath, dtype=str, keep_default_na=False)
cog_by_country_label = cnaf.build_country_cog_lookup(df_cog_pays)

df_qf_allocataires, unmapped_labels = cnaf.map_birth_country_to_cog(
    df_qf_allocataires, cog_by_country_label)
if unmapped_labels:
    print(f"{len(unmapped_labels)} PAYSNAIDOS label(s) without a COG match: {unmapped_labels}")

df_qf_allocataires, born_abroad_count = cnaf.clear_foreign_birthplace_insee(df_qf_allocataires)
print(f"{born_abroad_count} allocataire(s) born outside France: "
      "allocataire-code_insee_naissance cleared")

df_qf_batch_input = cnaf.select_qf_batch_columns(df_qf_allocataires)
df_qf_batch_input.to_csv(qf_batch_input_filepath, index=False, encoding='utf-8')

ars_rows = (df_psp_mapped_cnaf['situation_origine'] == 'ARS').sum()
unparsed_dob = df_qf_batch_input['allocataire-date_naissance'].isna().sum()
print(f"{len(df_qf_batch_input)} allocataire(s) (from {len(df_qf_route)} of {ars_rows} ARS row(s) "
      f"within the 6-17 ans window) written to {qf_batch_input_filepath} for qf-batch "
      f"({unparsed_dob} with an unparsed birthdate)")

## ⏸ Checkpoint: run qf-batch here
The cell above wrote `QF_BATCH_INPUT_PATHFILE_2026`. Run it through qf-batch.ts now
(detached, can take up to a week - see worker/src/scripts/qf-batch.ts), writing its
output to `QF_BATCH_OUTPUT_PATHFILE_2026`, before continuing past the next checkpoint
below. The cells in between only touch beneficiary-level data and don't need to wait.


In [ ]:
# remove unused
df_psp_mapped_cnaf = cnaf.drop_raw_address_columns(df_psp_mapped_cnaf)

In [ ]:
# remove rows with missing necessary values (if one of those value are missing we cannot
# generate a code), then columns with all null value
df_valid = cnaf.filter_rows_missing_required_fields(df_psp_mapped_cnaf)

In [ ]:
# Upper case these columns for the merge
df_valid = cnaf.normalize_identity_casing(df_valid)

In [ ]:
# lower case on emails on all
df_valid = cnaf.normalize_email_casing(df_valid)

In [ ]:
# Preliminary filter, ahead of the precise QF/AAH/AEEH windows below: 1996-01-01 is the
# oldest birthdate any of the 3 routes can accept (AAH's lower bound, cnaf.AAH_DOB_MIN).
df_valid_after = cnaf.filter_within_eligibility_floor(df_valid)

print(f"{len(df_valid) - len(df_valid_after)} rows removed because they are outside all eligibility windows")

In [ ]:
# add missing 0 to phone numbers, and set '0' phone values to None
df_valid_after = cnaf.fix_phone_number_formatting(df_valid_after)

In [ ]:
# set Nan values for not existing courriel
df_valid_after = cnaf.clear_blank_email(df_valid_after)

In [ ]:
# add 4h on all birthdates
df_valid_after = cnaf.shift_birthdate_by_hours(df_valid_after)

In [ ]:
# remove duplicate beneficiaries (see cnaf.DEDUPLICATION_KEY_COLUMNS)
df_valid_no_duplicate, duplicate_count = cnaf.drop_duplicate_beneficiaries(df_valid_after)

print(f"{duplicate_count} duplicate rows were removed")

In [ ]:
# map allocataire json
df_valid_no_duplicate = cnaf.add_allocataire_json_column(df_valid_no_duplicate)

In [ ]:
# map adresse_allocataire json
df_valid_no_duplicate = cnaf.add_adresse_allocataire_json_column(df_valid_no_duplicate)

In [ ]:
## drop the columns now folded into the JSON columns, plus the qf-batch pivot-only ones
## (see cnaf.FINAL_COLUMNS_TO_DROP)
df_final = cnaf.drop_intermediate_columns(df_valid_no_duplicate)

## ▶ Resume here once qf-batch has finished
The cell below reads `QF_BATCH_OUTPUT_PATHFILE_2026` and joins the verdict back onto
every beneficiary row of each allocataire.


In [ ]:
# qf-batch.ts runs out-of-band (can take up to a week) - read its verdict back in here.
# It records the raw quotient (qf_value), not an eligibility boolean: the threshold is
# applied below, in the QF route cell.
df_qf_batch_output = pd.read_csv(qf_batch_output_filepath, dtype=str, keep_default_na=False)
qf_value_by_allocataire = cnaf.build_qf_value_lookup(df_qf_batch_output)

# One quotient_familial value per household, fanned out to every child row of that
# allocataire. Rows without a value (404, error, non-ARS) stay NaN.
df_psp_mapped_cnaf = cnaf.attach_qf_value(df_psp_mapped_cnaf, qf_value_by_allocataire)

In [ ]:
# Quotient familial route: 6-17 ans révolus, household quotient must also clear the threshold
# (cnaf.QF_MAX). The same window already trimmed the qf-batch input above; re-applied here per
# beneficiary because an eligible household can also hold out-of-window children.
# The routes are computed on df_psp_mapped_cnaf, which alone carries qf_value, and select the
# matching rows of df_final.
df_final_jeune = cnaf.select_eligible_by_index(df_final, cnaf.qf_eligible_index(df_psp_mapped_cnaf))

In [ ]:
# AAH route: 16-30 ans révolus - situation already settled from CNAF's ORIGINESELECTION
df_final_aah = cnaf.select_eligible_by_index(df_final, cnaf.aah_eligible_index(df_psp_mapped_cnaf))

In [ ]:
# AEEH route: 6-19 ans révolus - no quotient_familial call needed, CNAF already flags it
df_final_aeeh = cnaf.select_eligible_by_index(df_final, cnaf.aeeh_eligible_index(df_psp_mapped_cnaf))

In [ ]:
# Merge QF, AAH and AEEH routes
df_final_jeune_and_aah = pd.concat(
    [df_final_jeune, df_final_aah, df_final_aeeh], ignore_index=True).reset_index(drop=True)


In [ ]:
df_final_jeune_and_aah.loc[:,'date_naissance'] = df_final_jeune_and_aah['date_naissance'].astype(str)

In [ ]:
# output to CSV files
df_final_jeune_and_aah.to_csv(base_output_filepath, sep=';', index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

In [ ]:
print(f"{len(df_final_jeune)} df_final_jeune")
print(f"{len(df_final_aah)} df_final_aah")
print(f"{len(df_final_aeeh)} df_final_aeeh")
print(f"{len(df_final_jeune_and_aah)} jeune, aah and aeeh")
